In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---
import matplotlib.image as mpimg

# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 6 &mdash; 2D Fourier Transforms</h1></center>

Geophysical data are rarely one-dimensional. A gravity or magnetic survey is a map, a seismic shot record is a function of time and receiver
position, and a GPR transect is an image. The 2D Fourier transform describes such data as a sum of plane waves, each with a wavelength and a
direction, so features with a common orientation or scale collect in one region of the wavenumber plane. That makes it possible to separate
signal from noise by direction and wavelength, for example slow ground roll from steep reflections, or shallow from deep gravity sources. In
this module we build intuition with photographs and simple patterns, define the 2D transform and its properties, compute the main 2D pairs
(rectangle, Gaussian, disk and impulse train), and then filter an image, a gravity map and a shot record.

# Outline
- [6.1 What Does an Image Look Like in the Fourier Domain?](#6.1-What-Does-an-Image-Look-Like-in-the-Fourier-Domain?)
- [6.2 A Simpler Example: Two Dipping Lines](#6.2-A-Simpler-Example:-Two-Dipping-Lines)
- [6.3 The 2D Fourier Transform](#6.3-The-2D-Fourier-Transform)
- [6.4 2D Fourier Basis Functions: Plane Waves](#6.4-2D-Fourier-Basis-Functions:-Plane-Waves)
- [6.5 Alternative Conventions](#6.5-Alternative-Conventions)
- [6.6 Similarities with the 1D Transform](#6.6-Similarities-with-the-1D-Transform)
- [6.7 Properties of the 2D Fourier Transform](#6.7-Properties-of-the-2D-Fourier-Transform)
- [6.8 2D Fourier Transform Pairs](#6.8-2D-Fourier-Transform-Pairs)
  - [6.8.1 Rectangle](#6.8.1-Rectangle)
  - [6.8.2 Gaussian](#6.8.2-Gaussian)
  - [6.8.3 Circular disk and the Airy pattern](#6.8.3-Circular-disk-and-the-Airy-pattern)
  - [6.8.4 2D impulse train](#6.8.4-2D-impulse-train)
- [6.9 Filtering an Image in the Wavenumber Domain](#6.9-Filtering-an-Image-in-the-Wavenumber-Domain)
- [6.10 A Potential-Field Map in the Wavenumber Domain](#6.10-A-Potential-Field-Map-in-the-Wavenumber-Domain)
- [6.11 Geophysical Filtering: Ground Roll](#6.11-Geophysical-Filtering:-Ground-Roll)
- [6.12 Parting Thoughts](#6.12-Parting-Thoughts)
- [Microquestions](#Microquestions)

<a name="6.1"></a>
## 6.1 What Does an Image Look Like in the Fourier Domain?

Let's begin this section by asking the following question: What would the 2D Fourier transforms of the images in Figure 6.1 look like?

In [ ]:
img1 = mpimg.imread('Fig/2.3_BRICKS.jpg').astype(float).mean(axis=2)      # average the three color channels
img2 = mpimg.imread('Fig/2.3_NEW_YORK.png').astype(float).mean(axis=2)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(9, 3.4), gridspec_kw=dict(width_ratios=[1, 1.5]))
axa.imshow(img1, cmap='gray'); axa.set_title('(a) Bricks')
axb.imshow(img2, cmap='gray'); axb.set_title('(b) New York at night')
for ax in (axa, axb):
    ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)'); ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 6.1.</b> Two examples of 2D "signals" whose spectral content we might examine or manipulate: (a) a brick wall (<a href="https://commons.wikimedia.org/wiki/File:Solna_Brick_wall_Stretcher_bond_variation1.jpg">Solna brick wall</a>) and (b) southern Manhattan at night (<a href="https://commons.wikimedia.org/wiki/File:Southern_end_of_Manhattan_at_night.jpg">Southern end of Manhattan at night</a>), both converted to grayscale. <i>(Wikimedia Commons.)</i>

Let's now look at these images' 2D Fourier magnitude spectra.  For now, we are going to calculate them in a way that is a bit more straightforward than the 1D transforms.  This is because the images we are dealing with are a function of **pixels**, which is not really a physical unit and thus it is harder to assign a meaning to what the Fourier components represent (other than cycles per pixel).  To calculate the 2D Fourier transform we can do the following:

    fft_img = np.fft.fft2(img)/(nx*ny)
    
where *nx* and *ny* are the dimensions of the image and are included as a normalization factor.  
 
 Just like in 1D Fourier Transforms, the output of the 2D DFT starts at $(u,v)=(0,0)$ cycles/pixel and runs to the largest positive frequencies half way along each axis. It then switches to the most negative frequencies and progresses back toward $(0,0)$.  Thus, we again need to use the *np.fft.fftshift()*  in order to put the 2D Fourier Transform of the image in order for them to make sense. In the images below we are going to look at the Fourier magnitude spectra. Thus, we will plot
 
     magnitude_spectrum = 20*np.log10(np.abs(np.fft.fftshift(fft_img))) 
 
where I have included the *20 np.log10()* term because the spectrum away from $(u,v)=(0,0)$ tends to be very small! Plotting it this way emphasizes the small values and shows how many decibels each component is down from the strongest. **Note that if you are plotting a 2D Fourier domain spectrum with the 20 np.log10() scaling already applied on it, you will have to undo this scaling if you want to take the inverse 2D Fourier transform.** 

Figure 6.2 shows the magnitude spectra of both images.

<div class="alert alert-warning">
<b>Common misconception:</b> "20 log" means base 10. Decibels use $20\log_{10}$; <code>np.log</code> is the natural logarithm, which stretches the dB scale by a factor of $\ln 10 = 2.303$. (The 2023 version of these notes used <code>20*np.log</code>.)
</div>

In [ ]:
def db_spectrum(img):
    """Centered 2D magnitude spectrum in dB relative to its maximum, with (u, v) axes in cycles/pixel."""
    F = np.abs(np.fft.fftshift(np.fft.fft2(img)))
    u = np.fft.fftshift(np.fft.fftfreq(img.shape[1])); v = np.fft.fftshift(np.fft.fftfreq(img.shape[0]))
    return 20*np.log10(F/F.max() + 1e-12), [u[0], u[-1], v[0], v[-1]]
fig, axs = plt.subplots(1, 2, figsize=(9, 4.0))
for ax, im, ttl in zip(axs, (img1, img2), ('(a) Bricks', '(b) New York at night')):
    S, ext = db_spectrum(im)
    h = ax.imshow(S, cmap='magma', origin='lower', extent=ext, vmin=-90, vmax=-20)
    ax.set_xlim(-0.2, 0.2); ax.set_ylim(-0.2, 0.2)        # zoom to where the energy is
    ax.set_title(ttl + ': spectrum', fontsize=12); ax.set_xlabel('$u$ (cycles/pixel)'); ax.set_ylabel('$v$ (cycles/pixel)'); ax.grid(False)
fig.colorbar(h, ax=axs, shrink=0.8, label='dB relative to the maximum')
plt.show()

<b>Figure 6.2.</b> Magnitude spectra of (a) the bricks and (b) New York at night, in dB relative to each maximum, with the zero frequency at the center (after <code>np.fft.fftshift</code>). The log scaling is for viewing only; undo it before any inverse transform. <i>(Python-generated.)</i>

<a name="6.2"></a>
## 6.2 A Simpler Example: Two Dipping Lines

While the above 2D FFTs are interesting and show a lot of structure, it's easier to start looking at something a little bit more straightforward.  In Figure 6.3 I have plotted on the left two dipping lines, one with brighter amplitudes at $45^\circ$ and the other with weaker amplitudes at $30^\circ$.  The right hand image shows the 2D Fourier transform of the figure on left.

In [ ]:
N = 256
iy, ix = np.mgrid[0:N, 0:N]                       # row (y) and column (x) indices
d30 = (iy - N/2) + 0.5*(ix - N/2)                 # distance-like coordinate across the weaker line
d45 = (iy - N/2) + (ix - N/2)                     # ... and across the brighter 45-degree line
img = np.exp(-(d30/np.sqrt(2))**2) + 2.5*np.exp(-(d45/np.sqrt(2))**2)
F = np.abs(np.fft.fftshift(np.fft.fft2(img))) / N**2
u = np.fft.fftshift(np.fft.fftfreq(N))
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 5))
axa.imshow(img, cmap='gray', origin='lower'); axa.set_title('(a) Two lines'); axa.set_xlabel('x (pixels)'); axa.set_ylabel('y (pixels)')
axb.imshow(F, cmap='magma', origin='lower', extent=[u[0], u[-1], u[0], u[-1]])
axb.set_title('(b) Magnitude spectrum'); axb.set_xlabel('$u$ (cycles/pixel)'); axb.set_ylabel('$v$ (cycles/pixel)')
for ax in (axa, axb): ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 6.3.</b> (a) Image of two lines with different dip and amplitude, and (b) its Fourier magnitude spectrum. <i>(Python-generated.)</i>

**Q1:** What do you notice about the relationship between dips between the 2D physical and 2D Fourier domains?

**Q2:** How does this relate to the Bricks and New York City at Night image examples above?

<a name="6.3"></a>
## 6.3 The 2D Fourier Transform

We define the forward **2D Fourier Transform** operator $\mathcal{F_{2D}}$ that acts on a 2D signal $h(x,y)$  according to

$$ 
\begin{eqnarray}
\mathcal{F_{2D}}\left[h(x,y)\right]=\widehat{H}(k_x,k_y) &=& 
\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} h(x,y) \,\mathrm{e}^{-i \mathbf{k}\cdot \mathbf{x}}\,\mathrm{d}x\,\mathrm{d}y\tag{1}\\
&=&
\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} h(x,y) \,\mathrm{e}^{-i (k_x x+k_y y)}\,\mathrm{d}x\,\mathrm{d}y\tag{2}
\end{eqnarray}
$$

where $\mathbf{x}=[x,y]$ are the two spatial variables (not the greatest notation) and $\mathbf{k}=[k_x,k_y]$ are the two Fourier conjugate domain variables and are termed the **spatial wavenumbers** that are equivalent to angular frequency.  You will also see the pair $[u,v]$, the spatial frequencies in cycles per unit length, with $\mathbf{k}=2\pi[u,v]$ (equation 8). The DFT routines in NumPy return $u$ and $v$, so the figures in this module use them.

The **2D Inverse Fourier Transform** operator, denoted $\mathcal{F}^{-1}_{2D}$, is given by:

$$ 
\begin{eqnarray}
\mathcal{F}^{-1}_{2D}\left[\widehat{H}(k_x,k_y)\right]=
h(x,y) &=& \frac{1}{4\pi^2} \int_{-\infty}^{\infty}\int_{-\infty}^{\infty} \widehat{H}(k_x,k_y)  \,\mathrm{e}^{i\mathbf{k}\cdot\mathbf{x}}\,\mathrm{d}k_x\,\mathrm{d}k_y \tag{3} \\
&=& \frac{1}{4\pi^2} \int_{-\infty}^{\infty}\int_{-\infty}^{\infty} \widehat{H}(k_x,k_y)  \,\mathrm{e}^{i(k_x x+k_y y)}\,\mathrm{d}k_x\,\mathrm{d}k_y
\tag{4} 
\end{eqnarray}
$$

You'll note that the scaling factor is given by $\frac{1}{2^N\pi^N}$ where $N$ is the dimensionality of the Fourier Transform.  Thus, a 5D Fourier Transform (yes, I've done one) would be scaled by $\frac{1}{32\pi^5}$.

<div class="alert alert-info">
<b>Convention:</b> This is the Module 5 convention extended to two dimensions: the forward kernel is $e^{-i\mathbf{k}\cdot\mathbf{x}}$ and the
inverse carries $1/(2\pi)$ for each dimension. Figures plot $u = k_x/2\pi$ and $v = k_y/2\pi$ in cycles per pixel (or per meter).
</div>

<a name="6.4"></a>
## 6.4 2D Fourier Basis Functions: Plane Waves

Probably not surprisingly, the basis functions for the 2D Fourier Transforms are oriented *cosine* and *sine* functions.  This is because

$$ \mathrm{e}^{\pm i \mathbf{k}\cdot\mathbf{x}} = \mathrm{e}^{\pm i \left(k_x x + k_y y\right)}= \mathrm{cos} \left( \mathbf{k}\cdot\mathbf{x}\right) \pm i\, \mathrm{sin}\left( \mathbf{k}\cdot\mathbf{x} \right)= \mathrm{cos} \left( k_x x + k_y y\right) \pm i\, \mathrm{sin}\left( k_x x + k_y y\right) \tag{5} $$

where the *sin* and *cos* are now **2D functions** that are related to [directional cosines](https://en.wikipedia.org/wiki/Direction_cosine) on the (x,y) plane.  The maxima and minima of, e.g., $\mathrm{cos}\left( \mathbf{k}\cdot\mathbf{x}\right)$ occur when 

$$\mathbf{k}\cdot\mathbf{x} = n\pi, \quad n=0,\pm 1, \pm 2, ..., \tag{6}$$

which are equally spaced lines with unit normal $\mathbf{k}/|\mathbf{k}|$ and wavelength $\lambda$ given by:

$$\lambda=\frac{2\pi}{|\mathbf{k}|}=\frac{2\pi}{\sqrt{k_x^2+k_y^2}} \tag{7}$$

Thus, when we talk about **longer wavelength features** (i.e., $\lambda\rightarrow \infty$), equation (7) indicates that we are looking at scenarios where both wavenumbers  approach zero (i.e., $k_x,k_y\rightarrow 0$).

Let's take a look at some of these 2D Fourier basis functions in Figure 6.4. Each wave fits a whole number of cycles $(m, n)$ into the $128\times128$ frame, so its spectrum is exactly two points.

In [ ]:
N = 128
yy, xx = np.mgrid[0:N, 0:N]
u = np.fft.fftshift(np.fft.fftfreq(N)); ext = [u[0] - 0.5/N, u[-1] + 0.5/N]*2
fig, axs = plt.subplots(2, 2, figsize=(9, 8.5))
for row, (m, n), lab in zip(axs, [(7, 4), (14, 8)], [('(a)', '(b)'), ('(c)', '(d)')]):
    h = np.cos(2*np.pi*(m*xx + n*yy)/N)
    H = np.abs(np.fft.fftshift(np.fft.fft2(h))) / N**2
    row[0].imshow(h, cmap='gray', origin='lower', extent=[0, N, 0, N])
    row[0].set_title(f'{lab[0]} cos 2π({m}x + {n}y)/{N}: λ = {N/np.hypot(m, n):.1f} px')
    row[0].set_xlabel('x (pixels)'); row[0].set_ylabel('y (pixels)')
    row[1].imshow(H, cmap='magma', origin='lower', extent=ext, vmin=0, vmax=0.5, interpolation='nearest')
    row[1].set_xlim(-0.16, 0.16); row[1].set_ylim(-0.16, 0.16)
    for s in (1, -1):
        row[1].annotate(f'({s*m}, {s*n})/{N}', xy=(s*m/N, s*n/N), xytext=(s*m/N - s*0.015, s*n/N + s*0.035),
                        color='white', fontsize=10, ha='center')
    row[1].set_title(f'{lab[1]} magnitude spectrum'); row[1].set_xlabel('$u$ (cycles/pixel)'); row[1].set_ylabel('$v$ (cycles/pixel)')
for ax in axs.flat: ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 6.4.</b> 2D cosines on a $128\times128$ grid. (a) $(m,n)=(7,4)$ cycles per frame: wavelength 15.9 px, wavevector azimuth 29.7°; (b) its magnitude spectrum, two points of height 0.5 at $\pm(7,4)/128$. (c) $(m,n)=(14,8)$: same direction, half the wavelength (7.9 px); (d) its spectrum, with the points twice as far from the origin. <i>(Python-generated.)</i>

You'll notice that the 2D Fourier transform of the basis function is like that of the 1D Fourier Transform - two $\delta$-functions but in opposing quadrants instead of positive/negative!  You'll also notice that the higher wavenumber magnitude $|{\bf k}|$ you go, the greater the separation between the delta function pair.

<div class="alert alert-info">
<b>Synthesis &mdash; complex exponentials as eigenfunctions (2D plane waves).</b> $e^{i\mathbf{k}\cdot\mathbf{x}}$ is the spatial part of the
PHGN 200 plane wave $E_0\cos(\mathbf{k}\cdot\mathbf{r}-\omega t)$: its crests are lines perpendicular to $\mathbf{k}$, a distance $2\pi/|\mathbf{k}|$
apart (the MATH 213 dot product picks out the direction). Along the direction of $\mathbf{k}$ it is exactly the 1D complex exponential of Modules 3
and 5; across that direction it is constant. A 2D spectrum is therefore a map of how much plane wave there is at each wavelength and each direction.
</div>

Figure 6.5 sweeps a plane wave through 30 directions and wavelengths. As the stripes rotate, the pair of points in the
spectrum rotates by the same angle; as the stripes get closer together, the points move away from the origin. The two points are always opposite each
other because the image is real (equation 18).

In [ ]:
N = 128                                         # 128 x 128 grid: integer (m, n) fine enough for a smooth, monotonic sweep
yy, xx = np.mgrid[0:N, 0:N]
n_frames = 30
az = np.radians(np.linspace(0, 170, n_frames)); K = np.linspace(8, 24, n_frames)
mn = []
for Kk, a in zip(K, az):                        # integer (m, n) pairs, so each wave fits the frame exactly (no leakage)
    p = (int(round(Kk*np.cos(a))), int(round(Kk*np.sin(a))))
    while p in mn or p == (0, 0):
        p = (p[0], p[1] + 1)
    mn.append(p)
u = np.fft.fftshift(np.fft.fftfreq(N)); ext = [u[0] - 0.5/N, u[-1] + 0.5/N]*2
fig, (axa, axb) = plt.subplots(1, 2, figsize=(8, 3.8))
im1 = axa.imshow(np.zeros((N, N)), cmap='gray', origin='lower', extent=[0, N, 0, N], vmin=-1, vmax=1, interpolation='nearest')
im2 = axb.imshow(np.zeros((N, N)), cmap='magma', origin='lower', extent=ext, vmin=0, vmax=0.5, interpolation='nearest')
axa.set_xlabel('x (pixels)'); axa.set_ylabel('y (pixels)'); axa.set_title('(a) plane wave')
axb.set_xlim(-0.22, 0.22); axb.set_ylim(-0.22, 0.22)
axb.set_xlabel('$u$ (cycles/pixel)'); axb.set_ylabel('$v$ (cycles/pixel)'); axb.set_title('(b) magnitude spectrum')
for ax in (axa, axb): ax.grid(False)
title = fig.suptitle('', fontsize=13)
state = {}

def update(k):
    m, n = mn[k]
    h = np.cos(2*np.pi*(m*xx + n*yy)/N)
    H = np.abs(np.fft.fftshift(np.fft.fft2(h))) / N**2
    im1.set_data(h); im2.set_data(H); state['H'] = H
    title.set_text(f'(m, n) = ({m}, {n}):  λ = {N/np.hypot(m, n):.1f} px,  wavevector azimuth = {np.degrees(np.arctan2(n, m)):.0f}°')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=300, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 6.5.</b> A rotating 2D plane wave on a $128\times128$ grid. (a) $\cos 2\pi(mx+ny)/128$ for 30 integer pairs $(m,n)$ whose wavevector azimuth runs from 0° to about 170° while $|(m,n)|$ grows from 8 to 24 cycles per frame (wavelength 16 to 5.3 pixels). (b) Its magnitude spectrum: two points at $\pm(m,n)/128$. <i>(Python-generated.)</i>

A few more examples with increasing complexity are shown in Figure 6.6.

In [ ]:
N = 128
yy, xx = np.mgrid[0:N, 0:N]
u = np.fft.fftshift(np.fft.fftfreq(N)); ext = [u[0] - 0.5/N, u[-1] + 0.5/N]*2
pa = np.cos(2*np.pi*(5*xx + 3*yy)/N)            # (a) cosine, (m, n) = (5, 3)
pc = np.sin(2*np.pi*(10*xx + 6*yy)/N)           # (c) sine, (10, 6): same direction, twice the wavenumber
pe = np.cos(2*np.pi*(8*xx - 5*yy)/N)            # (e) cosine with the opposite dip, (8, -5)
pg = pa + pe                                    # (g) sum of (a) and (e)
fig, axs = plt.subplots(2, 4, figsize=(12, 6.3))
labs = 'abcdefgh'
for i, (h, ttl) in enumerate([(pa, 'cos, (5, 3)'), (pc, 'sin, (10, 6)'), (pe, 'cos, (8, −5)'), (pg, '(a) + (e)')]):
    ax_s, ax_k = axs.flat[2*i], axs.flat[2*i + 1]
    H = np.abs(np.fft.fftshift(np.fft.fft2(h))) / N**2
    ax_s.imshow(h, cmap='gray', origin='lower', extent=[0, N, 0, N])
    ax_s.set_title(f'({labs[2*i]}) {ttl}', fontsize=12); ax_s.set_xlabel('x (pixels)'); ax_s.set_ylabel('y (pixels)')
    ax_k.imshow(H, cmap='magma', origin='lower', extent=ext, vmin=0, vmax=0.5, interpolation='nearest')
    ax_k.set_xlim(-0.12, 0.12); ax_k.set_ylim(-0.12, 0.12)
    ax_k.set_title(f'({labs[2*i + 1]}) magnitude spectrum', fontsize=12)
    ax_k.set_xlabel('$u$ (cycles/pixel)'); ax_k.set_ylabel('$v$ (cycles/pixel)')
for ax in axs.flat: ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 6.6.</b> Further 2D Fourier basis functions and their magnitude spectra on a $128\times128$ grid. (a) Cosine with $(m,n)=(5,3)$ and (b) its spectrum. (c) Sine with $(10,6)$: same direction, twice the wavenumber, and (d) its spectrum (a sine has the same magnitude spectrum as a cosine). (e) Cosine with the opposite dip, $(8,-5)$, and (f) its spectrum. (g) The sum of (a) and (e), and (h) its spectrum: the union of (b) and (f), by linearity. <i>(Python-generated.)</i>

Thus, in order to generate any of the images above, one has to know the 2D Fourier function $\widehat{H}(k_x,k_y)$, which provides the weights (i.e., the amount) of each 2D Fourier basis function to include to make $h(x,y)$ (e.g., in equation 3).

<a name="6.5"></a>
## 6.5 Alternative Conventions

Note that like in 1D there are different conventions for the 2D Fourier Transforms. For example if $[k_x,k_y]=2\pi[u,v]$ then you might see the following:

$$ 
\widehat{H}(u,v)=
\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} h(x,y) \,\mathrm{e}^{-i2\pi (u x+v y)}\,\mathrm{d}x\,\mathrm{d}y\tag{8} $$
and 
$$ 
h(x,y)=
\int_{-\infty}^{\infty}\int_{-\infty}^{\infty}\widehat{H}(u,v) \,\mathrm{e}^{i2\pi (u x+v y)}\,\mathrm{d}u\,\mathrm{d}v\tag{9} $$


Similarly, you might see the following symmetric definitions:

$$ 
\widehat{H}(k_x,k_y) 
=
\frac{1}{2\pi} \int_{-\infty}^{\infty}\int_{-\infty}^{\infty} h(x,y) \,\mathrm{e}^{-i (k_x x+k_y y)}\,\mathrm{d}x\,\mathrm{d}y\tag{10} $$
and
$$ 
h(x,y) = \frac{1}{2\pi} \int_{-\infty}^{\infty}\int_{-\infty}^{\infty} \widehat{H}(k_x,k_y)  \,\mathrm{e}^{i(k_x x+k_y y)}\,\mathrm{d}k_x\,\mathrm{d}k_y\tag{11} $$

Finally, because geophysicists are an interesting lot, you'll often see the following largely geophysics-specific definition for a mixed space-time Fourier Transform:
$$ 
\widehat{H}(\omega,k) 
=
\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} h(t,x) \,\mathrm{e}^{-i (\omega t-k x)}\,\mathrm{d}t\,\mathrm{d}x\tag{12} $$
and
$$ 
h(t,x) = \frac{1}{4\pi^2} \int_{-\infty}^{\infty}\int_{-\infty}^{\infty} \widehat{H}(\omega,k)  \,\mathrm{e}^{i(\omega t-k x)}\,\mathrm{d}\omega\,\mathrm{d}k \tag{13} $$


This is because the argument $\omega t-kx = const$ is often used to define a propagating wavefront.

<a name="6.6"></a>
## 6.6 Similarities with the 1D Transform

There are some significant similarities to the 1D scenario:

* FT pairs are denoted: 

$$h(x,y) \Leftrightarrow \widehat{H}(k_x,k_y) \tag{14} $$

* $\widehat{H}(k_x,k_y)$ is generally a complex quantity: 

$$\widehat{H}(k_x,k_y) = \widehat{H_R}(k_x,k_y)+i\widehat{H_I}(k_x,k_y) \tag{15}$$

* The real and positive-valued 2D power spectrum is given by: 

$$P\left[\widehat{H}(k_x,k_y)\right] = \left|\widehat{H}(k_x,k_y)\right|^2 = \overline{\widehat{H}(k_x,k_y)}\widehat{H}(k_x,k_y) \tag{16}$$

* The 2D phase spectrum is given by: 

$$\angle(\widehat{H}(k_x,k_y)) = \mathrm{atan2}\left(\widehat{H_I}(k_x,k_y),\,\widehat{H_R}(k_x,k_y)\right) \tag{17}$$

* Conjugacy: 

$$\overline{h(x,y)} \Leftrightarrow \overline{\widehat{H}(-k_x,-k_y)} \tag{18}$$

  so for a **real** image, $h=\overline{h}$ and $\widehat{H}(-k_x,-k_y)=\overline{\widehat{H}(k_x,k_y)}$: every spectral component has a mirror partner through the origin with the same magnitude.

* Symmetry: $h(x,y)$ is even about the $x=0$ and $y=0$ axis where 

$$h(x,y) = h(-x,y) \quad {\rm and} \quad h(x,y)= h(x,-y)\tag{19}$$

and $h(x,y)$ is odd about the $x=0$ and $y=0$ axis where 

$$h(x,y) = -h(-x,y) \quad {\rm and} \quad h(x,y)=-h(x,-y)\tag{20}$$

<a name="6.7"></a>
## 6.7 Properties of the 2D Fourier Transform

As in the 1D case, 2D FTs have a number of important properties. Given two 2D
functions, $f(x,y)$ and $g(x,y)$, that are acceptable for Fourier transform and have FTs given by $\widehat{F}(k_x,k_y)$ and $\widehat{G}(k_x,k_y)$,
the following properties hold.

**(1) Linearity.** If $h(x,y)=a f(x,y) + b g(x,y)$ where $a$ and $b$ are real numbers, show that the following is true:

$$ \widehat{H}(k_x,k_y) = a \widehat{F}(k_x,k_y) + b \widehat{G}(k_x,k_y) \tag{21} $$

<div class="alert alert-info">
<b>Proof:</b> $$
\begin{eqnarray}
\widehat{H}(k_x,k_y)&=&\mathcal{F}_{2D}\left[a f(x,y)+b g(x,y) \right] \\
&=& a \mathcal{F}_{2D}\left[f(x,y)\right] + b\mathcal{F}_{2D}\left[g(x,y)\right] \\
&=& a \widehat{F}(k_x,k_y) + b \widehat{G}(k_x,k_y) \tag{22}
\end{eqnarray}
$$

Note that the intermediate step is due to the <b>linearity</b> of the Fourier Transform.
</div>

**(2) Translation / shifting.** If $h(x,y) = f(x-a,y-b)$ where $a$ and $b$ are real numbers, show that the following is true:

$$ \widehat{H}(k_x,k_y)=\mathrm{e}^{-i(ak_x+bk_y)}\widehat{F}(k_x,k_y) \tag{23} $$

<div class="alert alert-info">
<b>Proof:</b> This follows essentially the same procedure as the 1D solution, but does it for two variables.

$$
\begin{eqnarray}
\widehat{H}(k_x,k_y) &=& \int_{-\infty}^{\infty} \int_{-\infty}^{\infty}f(x-a,y-b) \mathrm{e}^{-i(k_x x+k_y y)} \mathrm{d}x \,\mathrm{d}y \tag{24}\\
&=& \int_{-\infty}^{\infty} \int_{-\infty}^{\infty}f(x-a,y-b) \mathrm{e}^{-i(k_x x+k_y y)} \mathrm{e}^{-i(ak_x+bk_y)} \mathrm{e}^{i(ak_x+bk_y)} \mathrm{d}x \, \mathrm{d}y \tag{25}\\
&=& \mathrm{e}^{-i(ak_x+bk_y)}\int_{-\infty}^{\infty} \int_{-\infty}^{\infty}f(x-a,y-b) \mathrm{e}^{-i(k_x (x-a)+k_y (y-b))}  \mathrm{d}(x-a) \, \mathrm{d}(y-b) \tag{26} \\
&=& \mathrm{e}^{-i(ak_x+bk_y)}\int_{-\infty}^{\infty} \int_{-\infty}^{\infty}f(x^\prime,y^\prime) \mathrm{e}^{-i(k_x x^\prime+k_y y^\prime)} \mathrm{d}x^\prime \mathrm{d}y^\prime \tag{27} \\
&=& \mathrm{e}^{-i(ak_x+bk_y)} \widehat{F}(k_x,k_y)  \tag{28}\\
\end{eqnarray}
$$
</div>

**(3) Scaling theorem.** If $h(x,y) = f(ax,by)$ where $a$ and $b$ are real numbers and $a>0$ and $b>0$, show that the following is true:

$$ \widehat{H}(k_x,k_y) = \frac{1}{ab}\widehat{F}(\frac{k_x}{a},\frac{k_y}{b}) \tag{29} $$

For scale factors of either sign the factor becomes $1/|ab|$.

<div class="alert alert-info">
<b>Proof:</b> Again, similar to the solution for 1D, but with two variables. Let $x^\prime = ax$ and $y^\prime=by$, so that $k_x^\prime=k_x/a$ and $k_y^\prime=k_y/b$. Then we can write:

$$
\begin{eqnarray}
\widehat{H}(k_x,k_y) &=& \int_{-\infty}^{\infty} \int_{-\infty}^{\infty}f(ax,by) \mathrm{e}^{-i(k_x x+k_y y)} \mathrm{d}x \,\mathrm{d}y \tag{30}\\
&=& \int_{-\infty}^{\infty} \int_{-\infty}^{\infty}f(x^\prime,y^\prime) \mathrm{e}^{-i(k_x x^\prime/a+k_y y^\prime/b)} \mathrm{d}x^\prime \mathrm{d}y^\prime /(ab) \tag{31} \\
&=& \frac{1}{ab}\int_{-\infty}^{\infty} \int_{-\infty}^{\infty}f(x^\prime,y^\prime) \mathrm{e}^{-i(k^\prime_x x^\prime+k^\prime_y y^\prime)} \mathrm{d}x^\prime \mathrm{d}y^\prime \tag{32}\\
&=& \frac{1}{ab}\widehat{F}(k^\prime_x,k^\prime_y)=\frac{1}{ab}\widehat{F}(\frac{k_x}{a},\frac{k_y}{b}) \tag{33}
\end{eqnarray}
$$
</div>

<a name="6.8"></a>
## 6.8 2D Fourier Transform Pairs

Let's now examine a number of important 2D Fourier transform pairs: a rectangle, a Gaussian, a circular disk, and a 2D impulse train.

<a name="6.8.1"></a>
### 6.8.1 Rectangle

Consider a rectangle centered at the origin with half-widths $X$ and $Y$ (sides $2X$ and $2Y$) in the $x$ and $y$ dimensions, respectively:

$$f(x,y) = \left\{ \begin{array}{cl}
1, &\quad |x| \le X \,\,\,{\rm and}\,\,\, |y| \le Y \\
0, &\quad {\rm otherwise}
\end{array}
\right.\tag{34}$$

We can write the 2D Fourier transform as the following:

$$ 
\begin{eqnarray}
\widehat{F}(k_x,k_y) &=& 
\int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x,y)\, \mathrm{e}^{-i(k_x x+k_y y) } \mathrm{d}x \mathrm{d}y \tag{35}\\
\, &=& \int_{-X}^{X}\mathrm{e}^{-i k_x x }\mathrm{d}x \int_{-Y}^{Y}\mathrm{e}^{-i k_y y }\mathrm{d}y \tag{36}\\
\, &=& \left[ \frac{\mathrm{e}^{-ik_x x}}{-ik_x}  \right]_{-X}^{X}
       \left[ \frac{\mathrm{e}^{-ik_y y}}{-ik_y}  \right]_{-Y}^{Y} \tag{37}\\
\, &=& 4XY\left[\frac{\sin(k_x X)}{k_x X}\right] 
         \left[\frac{\sin(k_y Y)}{k_y Y}\right] \tag{38}\\ 
\, &=& 4XY \,\mathrm{sinc} \left(\frac{k_x X}{\pi}\right)\,\mathrm{sinc} \left(\frac{k_y Y}{\pi}\right) \tag{39}
\end{eqnarray}
$$

Here $\mathrm{sinc}(x)=\sin(\pi x)/(\pi x)$ is the normalized sinc used throughout the course, so in cycles per unit length the transform is $4XY\,\mathrm{sinc}(2uX)\,\mathrm{sinc}(2vY)$, with first zeros at $u=\pm 1/(2X)$ and $v=\pm 1/(2Y)$. The integral separates because the dependence on $x$ and $y$ is independent. Figure 6.7 shows the pair for a square with $X=Y=5$ pixels, so the first zeros are at $u, v = \pm 0.1$ cycles/pixel. Moving the square would multiply the spectrum by a phase factor (equation 23) but leave this magnitude unchanged.

In [ ]:
N = 256
img = np.zeros((N, N)); img[N//2 - 5:N//2 + 5, N//2 - 5:N//2 + 5] = 1      # centered square, half-width X = Y = 5 px
F = np.abs(np.fft.fftshift(np.fft.fft2(img))) / N**2
u = np.fft.fftshift(np.fft.fftfreq(N))
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 5))
axa.imshow(img, cmap='gray', origin='lower', extent=[-N/2, N/2, -N/2, N/2]); axa.set_xlim(-40, 40); axa.set_ylim(-40, 40)
axa.set_title('(a) Square, half-width 5 px'); axa.set_xlabel('x (pixels)'); axa.set_ylabel('y (pixels)')
axb.imshow(F, cmap='magma', origin='lower', extent=[u[0], u[-1], u[0], u[-1]])
for s in (-1, 1):
    axb.axvline(s*0.1, color='w', ls='--', lw=1); axb.axhline(s*0.1, color='w', ls='--', lw=1)
axb.set_title('(b) Magnitude spectrum; first zeros dashed'); axb.set_xlabel('$u$ (cycles/pixel)'); axb.set_ylabel('$v$ (cycles/pixel)')
for ax in (axa, axb): ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 6.7.</b> (a) A centered square of half-width 5 pixels and (b) its 2D magnitude spectrum, a product of two sinc functions with first zeros (dashed) at $u, v=\pm 0.1$ cycles/pixel. <i>(Python-generated.)</i>

<a name="6.8.2"></a>
### 6.8.2 Gaussian

A 2D Gaussian centered at the origin can be written as the following:

$$ f(x,y) = \mathrm{e}^{-r^2/(2\sigma^2)}= \mathrm{e}^{-(x^2+y^2)/(2\sigma^2)} \tag{40} $$

where $r^2=x^2+y^2$.  The 2D Fourier Transform of this function is given by:

$$\widehat{F}(k_x,k_y) = 2\pi\sigma^2 \mathrm{e}^{-\sigma^2 k^2/2} = 2\pi\sigma^2 \mathrm{e}^{-\sigma^2(k_x^2+k_y^2)/2} \tag{41}$$

where $k=|\mathbf{k}|=\sqrt{k_x^2+k_y^2}\ge 0$.  Again, you'll notice that the $\sigma$ in the space-domain expression is in the denominator of the exponential function, whereas it is in the numerator of the Fourier domain expression.  Thus, again, "squeezing" in one domain leads to "expansion" in the conjugate Fourier domain!

Figure 6.8 compares a narrow ($\sigma = 2$ px) and a broad ($\sigma = 8$ px) Gaussian.

In [ ]:
N = 256
y, x = np.mgrid[-N//2:N//2, -N//2:N//2]
u = np.fft.fftshift(np.fft.fftfreq(N))
fig, axs = plt.subplots(2, 2, figsize=(10, 9.2))
for row, sig, lab in zip(axs, (2.0, 8.0), (('(a)', '(b)'), ('(c)', '(d)'))):
    f = np.exp(-(x**2 + y**2)/(2*sig**2))
    F = np.abs(np.fft.fftshift(np.fft.fft2(f)))          # sum over pixels = continuous transform in pixel units
    row[0].imshow(f, cmap='gray', origin='lower', extent=[-N/2, N/2, -N/2, N/2]); row[0].set_xlim(-40, 40); row[0].set_ylim(-40, 40)
    row[0].set_title(f'{lab[0]} Gaussian, σ = {sig:.0f} px'); row[0].set_xlabel('x (pixels)'); row[0].set_ylabel('y (pixels)')
    h = row[1].imshow(F, cmap='magma', origin='lower', extent=[u[0], u[-1], u[0], u[-1]])
    row[1].set_xlim(-0.25, 0.25); row[1].set_ylim(-0.25, 0.25)
    row[1].set_title(f'{lab[1]} spectrum, peak 2πσ² = {2*np.pi*sig**2:.0f}'); row[1].set_xlabel('$u$ (cycles/pixel)'); row[1].set_ylabel('$v$ (cycles/pixel)')
    fig.colorbar(h, ax=row[1], shrink=0.85)
for ax in axs.flat: ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 6.8.</b> 2D Gaussians and their magnitude spectra: (a) $\sigma = 2$ px and (b) its spectrum; (c) $\sigma = 8$ px and (d) its spectrum. The spectrum is again a Gaussian, of peak $2\pi\sigma^2$ (the pixel sum of the image) and width $1/(2\pi\sigma)$ cycles/pixel: the broader Gaussian has the narrower, taller spectrum. <i>(Python-generated.)</i>

<a name="6.8.3"></a>
### 6.8.3 Circular disk and the Airy pattern

The [Airy pattern](https://en.wikipedia.org/wiki/Airy_disk#Mathematical_details) arises in [Fourier optics](https://en.wikipedia.org/wiki/Fourier_optics) as the diffraction pattern of a circular aperture. It is the transform of the circular disk of radius $a$ centered at the origin:

$$f(x,y) = \left\{
\begin{array}{c}
1, \quad |r|<a \\
0, \quad |r|\ge a \\
\end{array}
\right. \tag{42}
$$

With $k=|\mathbf{k}|\ge 0$, the 2D Fourier transform of this function is

$$\widehat{F}(k_x,k_y) = \frac{2\pi a}{k} \, J_1\left(ka\right)\tag{43}$$

where $J_1$ is a [Bessel Function](http://mathworld.wolfram.com/BesselFunctionoftheFirstKind.html) of the first kind (where the $1$ is the order of the Bessel function).  A derivation (via the Hankel transform) is given by Bracewell (see References). Its value at $k=0$ is the disk area $\pi a^2$, and its first zero is at $ka = 3.8317$, the first zero of $J_1$.  **Note that this function is the 2D cylindrical coordinate equivalent of the 1D sinc function!**

In [ ]:
N, a = 512, 40
y, x = np.mgrid[-N//2:N//2, -N//2:N//2]
disk = (np.hypot(x, y) < a).astype(float)
F = np.abs(np.fft.fftshift(np.fft.fft2(disk)))
S = 20*np.log10(F/F.max() + 1e-12)
u = np.fft.fftshift(np.fft.fftfreq(N))
u_ring = special.jn_zeros(1, 1)[0] / (2*np.pi*a)        # first zero of J1 at k a = 3.8317
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 5))
axa.imshow(disk, cmap='gray', origin='lower', extent=[-N/2, N/2, -N/2, N/2]); axa.set_xlim(-80, 80); axa.set_ylim(-80, 80)
axa.set_title(f'(a) Circular disk, radius a = {a} px'); axa.set_xlabel('x (pixels)'); axa.set_ylabel('y (pixels)')
axb.imshow(S, cmap='magma', origin='lower', extent=[u[0], u[-1], u[0], u[-1]], vmin=-70, vmax=0)
t = np.linspace(0, 2*np.pi, 200)
axb.plot(u_ring*np.cos(t), u_ring*np.sin(t), '--', color='cyan', lw=1.2)
axb.set_xlim(-0.1, 0.1); axb.set_ylim(-0.1, 0.1); axb.set_xticks([-0.1, -0.05, 0, 0.05, 0.1])
axb.set_title('(b) Airy pattern (dB); first dark ring dashed'); axb.set_xlabel('$u$ (cycles/pixel)'); axb.set_ylabel('$v$ (cycles/pixel)')
for ax in (axa, axb): ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 6.9.</b> (a) A circular disk of radius $a = 40$ px and (b) its 2D magnitude spectrum in dB, the Airy pattern. The first dark ring (dashed) is at $u = 3.8317/(2\pi a) = 0.0152$ cycles/pixel. <i>(Python-generated.)</i>

**Q:** Why is this not a perfect image of the 2D equivalent of the 1D sinc function?

<div class="alert alert-success">
<h3><center>Distraction A: A lens is a 2D Fourier transformer</center></h3>

The far-field (Fraunhofer) diffraction pattern of a circular aperture is the 2D Fourier
transform of the aperture, the Airy pattern of Figure 6.9. Its first dark ring is at $ka = 3.8317$. Converted to an angle, this gives the PHGN 200
resolution limit of a telescope or camera, $\theta \approx 1.22\,\lambda/D$, and $1.22 = 3.8317/\pi = 1.2197$.
</div>

<a name="6.8.4"></a>
### 6.8.4 2D impulse train

The 2D extension of the 1D impulse train (or Shah Function) is an important function for the digital sampling of continuous 2D fields.

In [ ]:
N, X, Y = 64, 4, 8
comb = np.zeros((N, N)); comb[::Y, ::X] = 1           # impulses every X = 4 px in x and Y = 8 px in y
C = np.abs(np.fft.fftshift(np.fft.fft2(comb))) / N**2
u = np.fft.fftshift(np.fft.fftfreq(N))
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 5))
yy, xx = np.nonzero(comb)
axa.plot(xx, yy, 'o', color=DSP_COLORS['sampled'], ms=4)
axa.set_xlim(-1, 33); axa.set_ylim(-1, 33); axa.set_aspect('equal')
axa.set_title(f'(a) 2D impulse train, X = {X} px, Y = {Y} px'); axa.set_xlabel('x (pixels)'); axa.set_ylabel('y (pixels)')
vv, uu = np.nonzero(C > 1e-9)
pts = [(a + du, b + dv) for a, b in zip(u[uu], u[vv]) for du in (0, 1) for dv in (0, 1) if a + du <= 0.5 and b + dv <= 0.5]
axb.plot(*np.array(pts).T, 'o', color=DSP_COLORS['freq_domain'], ms=6)   # the DFT is periodic: the -0.5 rows repeat at +0.5
axb.set_xlim(-0.55, 0.55); axb.set_ylim(-0.55, 0.55); axb.set_aspect('equal')
axb.set_title(f'(b) its 2D DFT: spacing 1/X = {1/X} and 1/Y = {1/Y}'); axb.set_xlabel('$u$ (cycles/pixel)'); axb.set_ylabel('$v$ (cycles/pixel)')
plt.tight_layout(); plt.show()

<b>Figure 6.10.</b> (a) A 2D impulse train with spacing $X = 4$ px in $x$ and $Y = 8$ px in $y$ (part of the $64\times64$ grid shown). (b) Its 2D DFT: impulses of height $1/(XY) = 0.03125$ spaced $1/X = 0.25$ cycles/pixel in $u$ and $1/Y = 0.125$ in $v$. The wider spacing in space gives the closer spacing in wavenumber. <i>(Python-generated.)</i>

In the space domain the 2D impulse train is

$$ \mathrm{III}_{X,Y}(x,y) = \sum_{m=-\infty}^{\infty} \sum_{n=-\infty}^{\infty} \delta(x-mX)\,\delta(y-nY) \tag{44} $$

Its 2D Fourier transform (in the convention of equation 1) is

$$ \mathcal{F}_{2D}\left[\mathrm{III}_{X,Y}(x,y)\right] = \frac{4\pi^2}{XY} \sum_{m=-\infty}^{\infty} \sum_{n=-\infty}^{\infty} \delta\!\left(k_x-\frac{2\pi m}{X}\right)\delta\!\left(k_y-\frac{2\pi n}{Y}\right) \tag{45} $$

In cycles per unit length this reads $\frac{1}{XY}\sum_m\sum_n \delta(u-m/X)\,\delta(v-n/Y)$: another impulse train, with spacing $1/X$ in $u$ and $1/Y$ in $v$ (Figure 6.10).

<div class="alert alert-info">
<b>Synthesis &mdash; sampling and periodization (the comb in two dimensions).</b> The 2D impulse train is a product of two 1D combs, so its transform is a
product of the two 1D comb transforms of Module 5, axis by axis. Spacing $X$ in $x$ gives spacing $2\pi/X$ in $k_x$, independently of $Y$ (Figure 6.10):
wider station spacing along one direction packs the spectral copies closer together along that axis only. Module 10 uses this pair to explain how map
and survey grids are sampled.
</div>

<a name="6.9"></a>
## 6.9 Filtering an Image in the Wavenumber Domain

To see why we look at 2D Fourier spectra, consider Figure 6.11. Panel (c) shows what happens if we reject everything in the spectrum of the bricks image outside a disk of radius 0.05 cycles/pixel, a low-pass filter that keeps 95.3% of the spectral energy. Panel (d) is the complement, a high-pass filter; adding (c) and (d) gives back (a) exactly.

In [ ]:
img = mpimg.imread('Fig/2.3_BRICKS.jpg').astype(float).mean(axis=2)
F = np.fft.fft2(img)
uu, vv = np.meshgrid(np.fft.fftfreq(img.shape[1]), np.fft.fftfreq(img.shape[0]))
mask = np.hypot(uu, vv) < 0.05                       # keep |(u, v)| < 0.05 cycles/pixel
low = np.real(np.fft.ifft2(F*mask)); high = np.real(np.fft.ifft2(F*~mask))
u = np.fft.fftshift(np.fft.fftfreq(img.shape[1])); ext = [u[0], u[-1], u[0], u[-1]]
dB = lambda G: 20*np.log10(np.abs(np.fft.fftshift(G))/np.abs(F).max() + 1e-12)
fig, axs = plt.subplots(1, 4, figsize=(13, 3.5))
t = np.linspace(0, 2*np.pi, 200)
axs[0].imshow(img, cmap='gray'); axs[0].set_title('(a) original')
axs[1].imshow(dB(F), cmap='magma', origin='lower', extent=ext, vmin=-90, vmax=-20)
axs[1].plot(0.05*np.cos(t), 0.05*np.sin(t), '--', color='cyan', lw=1.2)
axs[1].set_xlim(-0.2, 0.2); axs[1].set_ylim(-0.2, 0.2)
axs[1].set_title('(b) spectrum and mask'); axs[1].set_xlabel('$u$ (cycles/pixel)'); axs[1].set_ylabel('$v$ (cycles/pixel)')
axs[2].imshow(low, cmap='gray'); axs[2].set_title('(c) low-pass')
axs[3].imshow(high, cmap='gray'); axs[3].set_title('(d) high-pass')
for ax in axs[[0, 2, 3]]:
    ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
for ax in axs: ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 6.11.</b> 2D filtering by punching a disk out of the spectrum. (a) Original image. (b) Its magnitude spectrum (dB, central part) with the mask $|(u,v)| < 0.05$ cycles/pixel (dashed circle). (c) Low-pass image, rebuilt from the spectrum inside the mask. (d) High-pass image, rebuilt from the spectrum outside it; (c) + (d) = (a). <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; convolution and multiplication (sharp masks ring).</b> Multiplying the spectrum by a disk keeps the long wavelengths, but the product in the
wavenumber domain convolves the image with the transform of the disk, the Airy pattern of Figure 6.9. Its rings add faint ripples (ringing) around sharp
features of the low-passed image: the 2D version of the Gibbs overshoot from Module 4, for the same reason (a sharp cutoff). Tapering the mask edge reduces the
ripples at the cost of a less sharp cutoff.
</div>

<a name="6.10"></a>
## 6.10 A Potential-Field Map in the Wavenumber Domain

Gravity and magnetic maps are 2D signals governed by physics. Outside the
sources the gravitational potential $\phi$ satisfies Laplace's equation (GPGN 328). For the vertical gravity $g_z$ of a point mass $M$ at depth $z$ below
the observation plane, the 2D Fourier transform over $x$ and $y$ is

$$\widehat{g_z}(\mathbf{k}) = 2\pi G M\, \mathrm{e}^{-|\mathbf{k}| z} \tag{46}$$

The reason is Laplace's equation itself: transforming $\nabla^2\phi = 0$ over $x$ and $y$ turns $\partial_x^2+\partial_y^2$ into $-|\mathbf{k}|^2$, so

$$\frac{\partial^2 \widehat{\phi}}{\partial z^2} = |\mathbf{k}|^2\,\widehat{\phi} \tag{47}$$

an ordinary differential equation in $z$ (MATH 225) whose solution that decays away from the source is $\mathrm{e}^{-|\mathbf{k}|z}$. Moving the
observation plane up by $h$ therefore multiplies every component by the same kind of factor:

$$\widehat{g}(\mathbf{k}; z+h) = \mathrm{e}^{-|\mathbf{k}| h}\,\widehat{g}(\mathbf{k}; z) \tag{48}$$

This operation is called **upward continuation**. Figure 6.12 shows a map with two point masses, one shallow and one deep, chosen so that both peak
at the same 1.67 mGal.

In [ ]:
G = 6.674e-11                                  # gravitational constant (SI)
N, dx = 256, 50.0                              # 256 x 256 grid, 50 m spacing (12.8 km square)
xg = (np.arange(N) - N//2)*dx
X, Y = np.meshgrid(xg, xg)
def gz(M, x0, z):
    """Vertical gravity (m/s^2) of a point mass M (kg) at depth z (m) below (x0, 0)."""
    return G*M*z/((X - x0)**2 + Y**2 + z**2)**1.5
g1, g2 = gz(1e10, -2500, 200.0), gz(9e10, 2500, 600.0)
k = 2*np.pi*np.hypot(*np.meshgrid(np.fft.fftfreq(N, dx), np.fft.fftfreq(N, dx)))   # |k| in rad/m
up = np.real(np.fft.ifft2(np.fft.fft2(g1 + g2)*np.exp(-k*200.0)))                 # upward continuation by 200 m
fig, axs = plt.subplots(1, 3, figsize=(15, 4.8))
ext = [xg[0]/1e3, xg[-1]/1e3, xg[0]/1e3, xg[-1]/1e3]
for ax, im, ttl in ((axs[0], g1 + g2, '(a) $g_z$: sources at 200 m and 600 m'), (axs[2], up, '(c) (a) continued upward by 200 m')):
    h = ax.imshow(im*1e5, origin='lower', extent=ext, cmap='viridis', vmin=0, vmax=1.7)
    ax.set_xlabel('x (km)'); ax.set_ylabel('y (km)'); ax.set_title(ttl, fontsize=12); ax.grid(False)
fig.colorbar(h, ax=axs[2], shrink=0.85, label='mGal')
band = (k > 0.002) & (k < 0.012)
for gg, z, col in ((g1, 200, DSP_COLORS['time_domain']), (g2, 600, DSP_COLORS['freq_domain'])):
    A = np.abs(np.fft.fft2(gg))*dx**2
    sel = k < 0.02
    axs[1].plot(k[sel][::7], np.log(A[sel][::7]), '.', color=col, ms=3, alpha=0.5)
    p = np.polyfit(k[band], np.log(A[band]), 1)
    kk = np.linspace(0.002, 0.012, 2)
    axs[1].plot(kk, np.polyval(p, kk), color='k', lw=1.5)
    kx, ty = (0.013, np.polyval(p, 0.013) + 0.7) if z == 200 else (0.0008, -3.0)   # labels clear of both lines
    axs[1].text(kx, ty, f'slope = −{-p[0]:.0f} m', fontsize=11, va='center', color=col)
axs[1].set_xlim(0, 0.02); axs[1].set_xticks([0, 0.005, 0.01, 0.015, 0.02]); axs[1].set_xlabel('$|\\mathbf{k}|$ (rad/m)'); axs[1].set_ylabel(r'$\ln|\widehat{G}|$')
axs[1].set_title('(b) spectrum of each source alone', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 6.12.</b> (a) Vertical gravity of a $10^{10}$ kg point mass 200 m deep (left) and a $9\times10^{10}$ kg point mass 600 m deep (right), both peaking at 1.67 mGal, on a 256 × 256 grid with 50 m spacing. (b) $\ln|\widehat{G}|$ against $|\mathbf{k}|$ for each source alone (shallow blue, deep red); straight-line fits over 0.002–0.012 rad/m have slopes $-200$ m and $-599$ m, recovering the source depths of 200 m and 600 m (equation 46); the small misfit comes from the finite map. (c) Map (a) continued upward by 200 m: the shallow anomaly is weakened and broadened much more than the deep one. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; geophysical data as the running example (why gravity maps are smooth).</b> Every wavelength of a gravity map is attenuated by
$\mathrm{e}^{-|\mathbf{k}|z}$, in proportion to how short it is and how deep the source lies. That is why deep sources produce broad anomalies (the
inverse-square law of PHGN 100, seen in the wavenumber domain), why the slope of $\ln|\widehat{G}|$ against $|\mathbf{k}|$ measures depth
(Figure 6.12(b)), and why continuing a map upward is the same as multiplying its spectrum by $\mathrm{e}^{-|\mathbf{k}|h}$ (POTE1, GPGN 328).
</div>

<a name="6.11"></a>
## 6.11 Geophysical Filtering: Ground Roll

Figure 6.13 is a single shot gather extracted from a 2D land data set distributed with the Madagascar software. You will notice a number of different arrivals, including the direct wave, reflections and ground roll. How would you go about removing the unwanted ground roll?

<center><img src="Fig/2.3 GroundRoll.png" width="800"></center>

<b>Figure 6.13.</b> A 2D land seismic shot gather with strong reflections and slow, dispersive ground roll; the checkerboard texture in the ground-roll cone shows spatial aliasing. What strategy could you use to remove the unwanted ground roll? <i>(Source: field data distributed with the Madagascar open-source software package (GPL license), <a href="https://www.ahay.org">www.ahay.org</a>.)</i>

The 2D transform of a time–offset record uses the geophysical kernel of equation (12). A linear event moving across the spread at
apparent speed $v$, $h(t,x) = w(t - x/v)$, transforms to

$$\widehat{H}(\omega, k) = 2\pi\,\widehat{W}(\omega)\,\delta\!\left(k - \frac{\omega}{v}\right) \tag{49}$$

so all of its energy lies on the line $k = \omega/v$, or $f = v\,u$, through the origin of the $f$–$k$ plane. Figure 6.14 builds a synthetic gather with
ground roll and two reflections and shows where each lands.

In [ ]:
nx, dxs, dt, nt = 128, 10.0, 0.004, 1152        # 128 receivers every 10 m; 4 ms sampling, 4.6 s
t = np.arange(nt)*dt; xo = np.arange(nx)*dxs
ricker = lambda tt, f0: (1 - 2*(np.pi*f0*tt)**2)*np.exp(-(np.pi*f0*tt)**2)
gather = 3.0*ricker(t[:, None] - 0.1 - xo[None, :]/300.0, 8.0)                 # ground roll: 8 Hz, 300 m/s
for t0, v in ((0.4, 2000.0), (0.8, 2500.0)):                                     # two hyperbolic reflections, 25 Hz
    gather += ricker(t[:, None] - np.sqrt(t0**2 + (xo[None, :]/v)**2), 25.0)
# 2D transform with the kernel of equation (10): e^{-i omega t} in time, e^{+i k x} in space
H = np.fft.fft(np.fft.ifft(gather, axis=1)*nx, axis=0)
f = np.fft.fftshift(np.fft.fftfreq(nt, dt)); u = np.fft.fftshift(np.fft.fftfreq(nx, dxs))
S = 20*np.log10(np.abs(np.fft.fftshift(H))/np.abs(H).max() + 1e-12)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(10, 4.3))
axa.imshow(gather, cmap='gray', aspect='auto', extent=[0, xo[-1], t[-1], 0], vmin=-1, vmax=1)
axa.set_ylim(4.5, 0); axa.set_xlabel('offset x (m)'); axa.set_ylabel('time t (s)'); axa.set_title('(a) synthetic shot gather'); axa.grid(False)
axb.imshow(S, cmap='magma', aspect='auto', origin='lower', extent=[u[0], u[-1], f[0], f[-1]], vmin=-45, vmax=0)
uu = np.linspace(0, u[-1], 50)
axb.plot(uu, 300*uu, '--', color='cyan', lw=1.5, label='$f = v\\,u$, $v$ = 300 m/s')
axb.set_ylim(0, 45); axb.set_xlabel('$u$ (cycles/m)'); axb.set_ylabel('$f$ (Hz)'); axb.set_title('(b) f–k magnitude spectrum (dB)')
axb.legend(loc='upper left', fontsize=10); axb.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 6.14.</b> (a) Synthetic shot gather: 128 receivers every 10 m, 4 ms sampling, ground roll (8 Hz Ricker wavelet traveling at 300 m/s) and two hyperbolic reflections (25 Hz; zero-offset times 0.4 and 0.8 s; 2000 and 2500 m/s). (b) Its $f$–$k$ magnitude spectrum in dB. The ground roll lies along the line $f = 300\,u$ (dashed) and wraps around at the spatial Nyquist wavenumber 0.05 cycles/m above 15 Hz, while the reflections stay close to the $f$ axis. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; geophysical data as the running example (separating waves by speed).</b> In the $f$–$k$ plane, every event that moves across the
spread at apparent speed $v$ lies on the line $f = v\,u$ through the origin, because each frequency $f$ has wavelength $v/f$ (PHGN 200). Slow ground roll
(a few hundred m/s) sits far from the $f$ axis, while reflections, which arrive almost simultaneously across the spread, sit close to it. The two
overlap in frequency but not in direction, which is why an $f$–$k$ filter can remove ground roll where a frequency filter cannot (Microquestion 6.3).
</div>

<a name="6.12"></a>
## 6.12 Parting Thoughts

Understanding the structures of 2D Fourier Transforms - particularly, how different features like lines, disks and points map between the space and wavenumber domains is very important in geophysical data processing.  In particular, you may be looking to filter out certain types of coherent noise based on some Fourier attribute such as dip, spectral location, etc.  This is very hard to do without knowing where these types of noise fall in Fourier-land!

## Additional References

1. Bracewell, R.N., 1965, The Fourier Transform and Applications, McGraw-Hill, New York.
2. James, J.F., 2011, A Student's Guide to Fourier Transforms, 3rd ed, Cambridge University Press.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 6.1 &mdash; A lineament in an aeromagnetic grid</b>

An aeromagnetic grid with $\Delta x = \Delta y = 50$ m is dominated by one set of parallel ridges. Its 2D magnitude spectrum has exactly two bright points, at $(u, v) = \pm(0.0020,\,0.0015)$ cycles/m.

<b>(a)</b> Find the wavelength of the ridges and the azimuth of the wavevector (counterclockwise from $+x$). In which direction do the ridge crests run?

<b>(b)</b> The grid header was wrong: the true $x$ spacing is 50 m but the file says 25 m, so the processed map is $h(x,y) = f(2x, y)$. Use the scaling theorem, equation (29), to find where the points appear, and the apparent wavelength and azimuth.

<b>(c)</b> "The spectrum has two points, so the map contains two sets of ridges." Explain why this is wrong using equation (18).
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 6.1</b></summary>

<b>(a)</b> $|\mathbf{u}| = \sqrt{0.0020^2 + 0.0015^2} = 0.0025$ cycles/m, so $\lambda = 400$ m. Wavevector azimuth $\mathrm{atan2}(0.0015, 0.0020) = 36.9^\circ$; the crests are perpendicular to it, running at $126.9^\circ$ (equivalently $-53.1^\circ$).

<b>(b)</b> $\widehat{H}(k_x,k_y) = \tfrac12\widehat{F}(k_x/2, k_y)$, so a feature of $f$ at $u$ appears at $2u$: points at $\pm(0.0040,\,0.0015)$ cycles/m. Apparent $\lambda = 1/\sqrt{0.0040^2+0.0015^2} = 234$ m and azimuth $20.6^\circ$. Stretching one axis changes both the wavelength and the apparent direction, so a wrong grid spacing rotates lineaments.

<b>(c)</b> The map is real, so $\widehat{H}(-\mathbf{k}) = \overline{\widehat{H}(\mathbf{k})}$: every plane wave $\cos(\mathbf{k}\cdot\mathbf{x})$ contributes a conjugate pair of points at $\pm\mathbf{k}$ (Euler\'s formula, Module 3; MATH 213 vectors). Two points are one ridge set, as in Figure 6.4.
</details>

<div class="alert alert-warning">
<b>Microquestion 6.2 &mdash; Upward continuation is a low-pass filter</b>

The vertical gravity anomaly of a buried point mass at depth $z$ has the 2D transform $\widehat{g_z}(\mathbf{k}) = 2\pi GM\,\mathrm{e}^{-|\mathbf{k}|z}$ (Section 6.10, from Newton\'s law of gravitation in PHGN 100). Consider $z = 200$ m.

<b>(a)</b> By what factor is the spectrum at wavelength 500 m reduced relative to $|\mathbf{k}| = 0$? At 5000 m?

<b>(b)</b> For a different, unknown source, the radially averaged spectrum has amplitudes $A_1$ at $|\mathbf{k}| = 0.005$ rad/m and $A_2 = 0.0302\,A_1$ at $0.015$ rad/m. Estimate the source depth. Then show that upward-continuing a map by $h$ is equivalent to moving the source down by $h$.

<b>(c)</b> "Upward continuation by 200 m sharpens a gravity map, so it helps resolve shallow sources." What does it actually do to the 500 m and 5000 m wavelengths, and which operation would sharpen the map (and at what cost)?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 6.2</b></summary>

<b>(a)</b> $k = 2\pi/\lambda$. For $\lambda = 500$ m, $k = 0.01257$ rad/m and the factor is $\mathrm{e}^{-0.01257\times200} = 0.081$. For $\lambda = 5000$ m, $k = 0.001257$ rad/m and the factor is $0.778$.

<b>(b)</b> $\ln(A_2/A_1) = -(k_2-k_1)z$, so $z = -\ln(0.0302)/0.010 = 350$ m. Continuation multiplies by $\mathrm{e}^{-|\mathbf{k}|h}$ (equation 48), and $\mathrm{e}^{-|\mathbf{k}|z}\,\mathrm{e}^{-|\mathbf{k}|h} = \mathrm{e}^{-|\mathbf{k}|(z+h)}$: the same spectrum as a source at depth $z + h$. The exponential solution comes from the MATH 225 ODE of equation (47).

<b>(c)</b> It multiplies every wavelength by $\mathrm{e}^{-|\mathbf{k}|h} \le 1$: the 500 m component drops to 8.1% while the 5000 m component keeps 77.8%. That is a smoothing (low-pass) operation that suppresses shallow sources, as in Figure 6.12(c). Downward continuation, $\mathrm{e}^{+|\mathbf{k}|h}$, sharpens the map, but it amplifies short-wavelength noise exponentially and must be stabilized.
</details>

<div class="alert alert-warning">
<b>Microquestion 6.3 &mdash; Ground roll in the f–k plane</b>

A land shot record has receivers every $\Delta x = 10$ m and $\Delta t = 4$ ms. Ground roll travels at 300 m/s with energy between 5 and 12 Hz. Reflections have apparent velocities above 2500 m/s and energy between 10 and 40 Hz.

<b>(a)</b> Using $v = f\lambda$ (PHGN 200), find the range of $u = 1/\lambda$ (cycles/m) occupied by the ground roll, and the largest $|u|$ of the reflections at 10 Hz and 40 Hz.

<b>(b)</b> Using the kernel of equation (12) and the 1D translation theorem (Module 5), show that the linear event $h(t,x) = w(t - x/v)$ has the 2D transform $2\pi\widehat{W}(\omega)\,\delta(k - \omega/v)$. What shape does it have in the $f$–$u$ plane?

<b>(c)</b> "To remove the ground roll, just apply a 15 Hz low-cut filter." What does this do to the reflections, and why does an $f$–$k$ filter that rejects apparent velocities below 1000 m/s do better?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 6.3</b></summary>

<b>(a)</b> Ground roll: $u = f/v$ from $5/300 = 0.0167$ to $12/300 = 0.040$ cycles/m (wavelengths 60 m to 25 m). Reflections: $|u| \le f/2500$, which is 0.004 cycles/m at 10 Hz and 0.016 at 40 Hz.

<b>(b)</b> $\iint w(t - x/v)\,\mathrm{e}^{-i\omega t}\,\mathrm{e}^{ikx}\,\mathrm{d}t\,\mathrm{d}x$: the $t$ integral is $\widehat{W}(\omega)\,\mathrm{e}^{-i\omega x/v}$ (translation theorem), leaving $\widehat{W}(\omega)\int \mathrm{e}^{i(k-\omega/v)x}\,\mathrm{d}x = 2\pi\widehat{W}(\omega)\,\delta(k - \omega/v)$. All of its energy lies on the straight line $k = \omega/v$, i.e. $f = v\,u$, through the origin; the slower the event, the flatter the line (Figure 6.14).

<b>(c)</b> A 15 Hz low-cut removes all of the ground roll but also the 10–15 Hz part of the reflection band. The two overlap in frequency, not in wavenumber: at 10 Hz the ground roll sits at $u = 0.033$ cycles/m and the reflections at $|u| \le 0.004$. Rejecting $|f/u| < 1000$ m/s removes the ground-roll line and leaves the steep reflection wedge, including its low frequencies.
</details>